In [1]:
import os, sys
import numpy as np
import pandas as pd
import re
import pickle
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, balanced_accuracy_score
import random
from datetime import date
import time
from sklvq import GLVQ
import sklvq
parts=os.getcwd().split('/')[:-1]
datapath='/'.join(parts)+'/data/'

dname_all=['diabetes', 'surgical', 'banking', 'adult', 'criteo']
dname=dname_all[2]#'adult'
#dname='breastcancer'
#trainset, testset=pd.read_csv(datapath+'%s_trainset.csv'%dname), pd.read_csv(datapath+'%s_testset.csv'%dname)
#unlearnset=pd.read_csv(datapath+'unlearn_samples.csv')
#features=trainset.columns[:-1]
code_path='/'.join(parts)+'/scripts/'
sys.path.append(code_path)
from experiment_utils import dataset_health
Xtrain, Ytrain, Xtest, Ytest, features=dataset_health(dname)
print(features)
if dname=='adult':
    features=['age', 'num-sex', 'race-summary', 'education-num', 'num-marital',
       'fnlwgt', 'capital-gain', 'hours-per-week', 'Reg_US', 'Work_Private']
elif dname=='diabetes':
    features=['num_gender','num_cat_age', 'num_change', 'num_diabetesMed', 'time_in_hospital',
       'num_lab_procedures', 'num_procedures', 'num_medications','number_outpatient', 
       'number_emergency', 'number_inpatient','number_diagnoses', 'num_acarbose', 'num_acetohexamide', 
       #'num_insulin','num_chlorpropamide', 'num_citoglipton', 'num_examide', 'num_glimepiride',
       #'num_pioglitazone','num_glipizide', 'num_metformin', 'num_glyburide',  'num_rosiglitazone',
     #  'num_glimepiride-pioglitazone', 'num_metformin-pioglitazone', 'num_glipizide-metformin', 
     #  'num_glyburide-metformin', 'num_metformin-rosiglitazone', 'num_miglitol', 'num_nateglinide', 
       #'num_repaglinide', 'num_tolazamide', 'num_tolbutamide', 'num_troglitazone',
       'Disch_Home/hospice', 'AfricanAmerican', 'Caucasian']
elif dname=='surgical':
    features=['bmi', 'Age','gender','race', 'asa_status', 'baseline_cancer', 'baseline_charlson',
       'baseline_cvd', 'baseline_dementia', 'baseline_diabetes',
       'baseline_digestive', 'baseline_osteoart', 'baseline_psych',
       'baseline_pulmonary', 'ahrq_ccs', 'ccsComplicationRate',#'ccsMort30Rate',
       'complication_rsi', #'dow', 'hour', 'month',
       #'moonphase', 'mort30', 
        'mortality_rsi',
    ]
    
else:
    print('No feature preset reqd')
%load_ext autoreload
#parts=os.getcwd().split('/')
#code_path='/'.join(parts[:parts.index("toolboxes")+1])+'/sklvq/'
#sys.path.append(code_path)
#from sklvq import GMLVQ, LGMLVQ
Xtrain.head(4)

/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq
Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed',
       'num_education', 'num_loan', 'num_housing', 'num_contact',
       'num_poutcome', 'num_default', 'num_job', 'bin_marital'],
      dtype='str')
No feature preset reqd


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,num_education,num_loan,num_housing,num_contact,num_poutcome,num_default,num_job,bin_marital
0,40,94,2,999,0,1.4,93.918,-42.7,4.960,5228.1,3,0,1,1,-1,-1,1.0,1.0
1,31,116,4,999,0,-1.8,92.893,-46.2,1.244,5099.1,6,0,0,0,-1,0,0.0,1.0
2,59,13,6,999,1,-1.8,92.893,-46.2,1.354,5099.1,1,0,0,0,0,0,5.0,1.0
3,43,94,5,999,0,1.4,93.918,-42.7,4.961,5228.1,3,0,1,0,-1,0,3.0,0.0


## Training of original training set

### Data normalization



In [32]:
%autoreload 2
from experiment_utils import data_normalization, data_norm_log
#zXtrain, zXtest=data_norm_log(Xtrain[features], Xtest[features])
zXtrain, zXtest=data_norm_log(Xtrain, Xtest)
print(zXtrain.shape, zXtest.shape, np.unique(Ytrain))

(81412, 43) (20354, 43) [0 1]


### Random selection of 'n' samples to unlearn

### Samples to unlearn, relearn, and normalization of samples to relearn

- Samples to be unlearned are selected as follows
     * Randomly selected 'n' samples, where n $\in [ 1,5,10,20,30 ]$
     * Select `n' samples that are most distant from eaither or both prototypes

- Relearning is retraining of model on original data minus the samples to be unlearned


In [33]:
from utils import samples_unlearn_random
from collections import Counter
if dname=='breastcancer':
    n=200
else:
    n=1000
print(Counter(Ytrain), n/len(Ytrain))
random_learn_set=samples_unlearn_random(Xtrain[features],Ytrain, n,0)
unlearn_indices,relearn_indices=random_learn_set['unlearn_indices'], random_learn_set['relearn_indices']
unlearn_samples,relearn_samples=random_learn_set['unlearn_samples'], random_learn_set['relearn_samples']
unlearn_labs,relearn_labs=random_learn_set['unlearn_labs'], random_learn_set['relearn_labs']
#zXretrain, zXretest=data_norm_log(Xtrain[features].iloc[relearn_indices], Xtest[features])
zXretrain=zXtrain.iloc[relearn_indices]
#zXretrain, zXretest=(relearn_samples-mu_new)/std_new, (Xtest-mu_new)/std_new

Counter({0: 43912, 1: 37500}) 0.012283201493637302


### Training Generalized LVQ models on the original training get 
- A copy of it is also created to measure deviation from original model after unlearning has been accomplished
- Set seed to ensure the copy of original that is made is a true identical copy, for reproducibility and comparability

In [36]:
dist_name, activation_type="squared-euclidean", "identity"
solver_type="wgd" #"sgd"
solver_params={"max_runs": 5, "step_size": np.array([0.05]), "k": 3,
              }
nprots_per_class=2
if solver_type in ["sgd", "wgd"]:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
else:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, random_state=42)
glvq.fit(zXtrain, Ytrain)
glvq_copy.fit(zXtrain[features], Ytrain)
est_train, est_test=glvq.predict(zXtrain), glvq.predict(zXtest)
est_train_copy, est_test_copy=glvq_copy.predict(zXtrain[features]), glvq_copy.predict(zXtest[features])
print(zXtrain.shape, len(features))
print('Whole Train: ',accuracy_score(Ytrain, est_train), balanced_accuracy_score(Ytrain, est_train))
print('Whole Test: ',accuracy_score(Ytest, est_test), balanced_accuracy_score(Ytest, est_test))

print('Ftr subset Train: ',accuracy_score(Ytrain, est_train_copy), balanced_accuracy_score(Ytrain, est_train_copy))
print('Ftr subset Test: ',accuracy_score(Ytest, est_test_copy), balanced_accuracy_score(Ytest, est_test_copy))

(81412, 43) 17
Whole Train:  0.6171571758463126 0.6089014501730734
Whole Test:  0.6161442468310897 0.608226549563068
Ftr subset Train:  0.6170343438313762 0.6087895330054047
Ftr subset Test:  0.6151125085978186 0.6069893432990416


In [59]:
nprots_per_class=2
if solver_type in ["sgd", "wgd"]:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
else:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, random_state=42)
glvq.fit(zXtrain[features], Ytrain)
est_train, est_test=glvq.predict(zXtrain[features]), glvq.predict(zXtest[features])
print(zXtrain.shape, len(features))
print('Train: ',accuracy_score(Ytrain, est_train), balanced_accuracy_score(Ytrain, est_train))
print('Test: ',accuracy_score(Ytest, est_test), balanced_accuracy_score(Ytest, est_test))

(81412, 43) 31
Train:  0.5966810789564192 0.5704004445254145
Test:  0.5963938292227572 0.5711404964270226


### Retraining on partial data (only relearn_samples) and unlearning from original trained model

In [52]:
%autoreload 2
from unlearning.unlearn_eval import *
# Retraining of GLVQ model with all params excatly the same as teh original model
dev0, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
print('Before unlearning: Deviation between original model and its copy:', dev0)
st=time.time()
glvq_partial1=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type#, solver_params=solver_params
)
glvq_partial1.fit(zXretrain, relearn_labs)
elapsed_retrain=(time.time()-st)/60
est_partial_retrain, est_partial_retest=glvq_partial1.predict(zXretrain), glvq_partial1.predict(zXtest)
glvq_copy.secure_prototypes_=glvq.prototypes_.copy()

print('Train: ',accuracy_score(relearn_labs, est_partial_retrain), balanced_accuracy_score(relearn_labs, est_partial_retrain))
print('Test: ',accuracy_score(Ytest, est_partial_retest), balanced_accuracy_score(Ytest, est_partial_retest))

Before unlearning: Deviation between original model and its copy: 0.0
Train:  0.6175731801338009 0.6082577261989497
Test:  0.6160459860469687 0.6069987537426179


### Unlearning from copy of original model, and comparing model characteristics and performance
* Characteristics comparison: Fidelity of prototypes
* Performance comparison: Raw deviation in predictions, deviation in Balanced accuracy and ROC-AUC

In [31]:
%autoreload 2
from collections import Counter
from unlearning.unlearn_eval import compare_perf
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq
training_info={'setsize':zXtrain.shape[0], 'class_weight':Counter(Ytrain)}
glvq_copy.prototypes_=glvq.prototypes_.copy()
glvq_copy.unlearn_rate_=0.01
st_un=time.time()
updated_model=unlearn_sample_effect_glvq(glvq_copy, zXtrain.iloc[unlearn_indices], unlearn_labs, training_info)
elapsed_untrain=(time.time()-st_un)/60
#glvq_copy.set_prototypes(updated_prots
dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
print('After unlearning: Deviation between original model and its unlearned copy:', dev02)
dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1, updated_model)
print('After unlearning: Deviation between retrained and unlearned moodels:', dev12)
dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_partial1)
print('After retraining: Deviation between original and retrained model:', dev01)
cratio_ur_uo=dev12/dev02
print('dev(prots from retrained and unlearned models) /dev(prots from original and unlearned models)=%0.03f'%cratio_ur_uo)
data_dict={'zX_M1':zXtest}
dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
print(elapsed_retrain, elapsed_untrain)
print('Train: ',accuracy_score(relearn_labs, updated_model.predict(zXretrain)), 
      balanced_accuracy_score(relearn_labs, updated_model.predict(zXretrain), adjusted=False))
print('Test: ',accuracy_score(Ytest, updated_model.predict(zXtest)), 
      balanced_accuracy_score(Ytest, updated_model.predict(zXtest), adjusted=False))

After unlearning: Deviation between original model and its unlearned copy: 0.5913688254952428
After unlearning: Deviation between retrained and unlearned moodels: 0.642718320438347
After retraining: Deviation between original and retrained model: 0.19521215360357833
dev(prots from retrained and unlearned models) /dev(prots from original and unlearned models)=1.087
0.005545393625895182 0.0013902862866719565
Train:  0.6381218038121804 0.6710440435794176
Test:  0.6351212845917321 0.6775997227135203


In [33]:
import scipy.spatial.distance#.jensenshannon as jensenshannon
import scipy
print('Divergence')
for c in range(0,len(glvq.prototypes_labels_)):
   # print()
    print(rmse(glvq.prototypes_[c], updated_model.prototypes_[c])-rmse(glvq_partial1.prototypes_[c], updated_model.prototypes_[c]),
         rmse(glvq.prototypes_[c], glvq_partial1.prototypes_[c]))
    print(kl_divergence(glvq.prototypes_[c], updated_model.prototypes_[c]),
         kl_divergence(glvq_partial1.prototypes_[c], updated_model.prototypes_[c]),
         kl_divergence(glvq.prototypes_[c], glvq_partial1.prototypes_[c]))

from sklearn.metrics.cluster import normalized_mutual_info_score, adjusted_mutual_info_score
print('Train')
print(adjusted_mutual_info_score(updated_model.predict(zXretrain),glvq.predict(zXretrain)))
print(adjusted_mutual_info_score(updated_model.predict(zXretrain),glvq_partial1.predict(zXretrain)))
print(adjusted_mutual_info_score(glvq_partial1.predict(zXretrain),glvq.predict(zXretrain)))
print('Unlearned samples')
print(adjusted_mutual_info_score(updated_model.predict(zXtrain.iloc[unlearn_indices]),glvq.predict(zXtrain.iloc[unlearn_indices])))
print(adjusted_mutual_info_score(updated_model.predict(zXtrain.iloc[unlearn_indices]),glvq_partial1.predict(zXtrain.iloc[unlearn_indices])))
print(adjusted_mutual_info_score(glvq_partial1.predict(zXtrain.iloc[unlearn_indices]),glvq.predict(zXtrain.iloc[unlearn_indices])))



Divergence
-0.005935285372361321 0.016976118141444358
nan nan nan
-0.0031937002540746162 0.010730895414243422
nan nan nan
-1.397293948267345 2.6879569675497224
nan nan nan
-1.8337601636202403 2.697191340635642
nan nan nan
Train
0.14382779609396815


/tmp/ipykernel_300522/381087277.py:4: RuntimeWarning: invalid value encountered in log
  div=np.sum(p*np.log(p)-p*np.log(q))
/tmp/ipykernel_300522/381087277.py:4: RuntimeWarning: invalid value encountered in log
  div=np.sum(p*np.log(p)-p*np.log(q))
/tmp/ipykernel_300522/381087277.py:4: RuntimeWarning: invalid value encountered in log
  div=np.sum(p*np.log(p)-p*np.log(q))
/tmp/ipykernel_300522/381087277.py:4: RuntimeWarning: invalid value encountered in log
  div=np.sum(p*np.log(p)-p*np.log(q))


0.14212015232069466
0.9740597432445461
Unlearned samples
0.13652828979387563
0.13509381748564858
0.9770395440439547


In [52]:
print(rmse(updated_model.prototypes_, glvq_partial1.prototypes_))
print(rmse(updated_model.prototypes_, glvq.prototypes_))
print(rmse(glvq.prototypes_, glvq_partial1.prototypes_))
c=0
print(updated_model.prototypes_[c])
print(glvq_partial1.prototypes_[c])
print(glvq.prototypes_[c])
mul
glvq_copy.prototypes_labels_

3375.326644742398
4889.149659226517
2061.8656865839125
[  5.97023715   7.14365182  -6.12209701 -12.46175766  -9.90757012
  -5.76282541 -15.93938709 -14.05133647 -13.25388883  -9.38389233
 -14.76660734 -14.48274962  -3.35665656  -4.12701883 -10.64667634
 -14.85080259  -5.79404548  -6.94762699   3.96432116   1.58464002
  -6.80943667 -15.9193375  -13.9776929   -1.76559186]
[  3.3177411    4.57922074  -1.25841241  -9.23202995  -9.85627625
   1.55730537  -9.15232064  -9.9169439  -10.24245301   1.11358086
  -8.51691248  -9.24632892 -10.93179168  -2.63008986  -5.79566287
  -8.88961772  -8.40341617  -0.89010488   2.20601665   2.8696745
  -7.01838284  -9.10826557  -9.63048489   0.74107333]
[  3.49765282   3.76929082  -7.51008585  -7.07704065  -5.99781303
  -5.36164904  -9.00585819  -6.58010264  -6.49741425  -8.19684865
  -6.96618689  -7.77229838   3.04202296  -2.71609026  -6.73040354
 -10.89479868  -1.17473163  -6.06417174   2.28792449   1.46596979
  -1.40522033  -9.92000647 -11.03397013  -1.67

array([0, 0, 1, 1])

In [46]:
print(glvq.prototypes_[0])


[-3.93552203e+00 -5.29779865e+00 -1.03987766e-04  8.70919910e-01
  3.29287836e+00 -1.61335405e+00  2.44102728e+00 -1.06441468e+01
 -1.10451908e+01 -1.47917000e+01  1.58582163e+00 -9.23968448e+00
 -9.21179151e+00  3.96752997e+00 -9.21022415e+00 -9.21034038e+00
 -9.21034037e+00 -8.60234838e+00 -9.21154630e+00 -8.32682334e+00
 -9.21215095e+00 -7.87980840e+00 -9.11891544e+00 -3.62621168e+00
 -5.61795530e+00 -9.20907953e+00 -9.20757880e+00 -9.21811089e+00
 -9.15405797e+00 -8.59060650e+00 -9.23028569e+00 -8.67662170e+00
 -9.20354687e+00 -9.20291444e+00 -9.21154769e+00 -8.94131092e+00
 -1.86544974e+00 -9.16131195e+00 -8.76654838e+00 -8.10674050e+00
 -7.39392751e+00 -3.35570639e+00 -8.52865361e+00]


In [34]:
ind_outlier_ind=np.where(np.abs(dist_diff-dist_same)<=1000)[0]
print(norm_dist_same[ind_outlier_ind], norm_dist_diff[ind_outlier_ind])
print(dist_same[ind_outlier_ind], dist_diff[ind_outlier_ind])
#np.sum([dist_same, dist_diff], axis=0)

[] []
[] []


In [35]:
%autoreload 2
from utils import relearn_unlearn_samples
dist_name, activation_type="squared-euclidean", "identity"
solver_type, solver_params="sgd", {"max_runs": 5, "step_size": np.array([0.05]),  #"k": 3,
                                  }
nprots_per_class=2
glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)

glvq.fit(zXtrain, Ytrain)
training_info={'setsize':zXtrain.shape[0], 'class_weight':Counter(Ytrain)}
########################################################################################
dist_func=sklvq.distances.SquaredEuclidean()
rel_distances=dist_func(zXtrain, glvq)
normed_distances=rel_distances/rel_distances.sum(axis=1)[:,np.newaxis]
sorted_dist, sorted_ind=np.sort(normed_distances, axis=1), np.argsort(normed_distances, axis=1)
closest_dists=(sorted_dist[:,1] -sorted_dist[:,0])
#thresh_opts=[0.000001,0.00001, 0.0001,0.001,0.01]
if dname=='criteo':
    nopts=[0.0001,0.001,0.01,0.1]
else:
    nopts=[0.0001,0.001,0.01,0.05,0.1]
########################################################################################
# Unlearning parameters to compare
# * number of random samples to unlearn n=[1,5,20,30,50]
cint=0
for n in nopts:
    unlearn_indices=np.where(closest_dists<=n)[0]
    relearn_indices, relearn_samples=relearn_unlearn_samples(Xtrain, unlearn_indices,0)
    if (len(unlearn_indices)==0)|(len(relearn_indices)==0):
        print(n, len(unlearn_indices))
        continue
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy.fit(zXtrain, Ytrain)
    dev00, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
    print('Before unlearning: Deviation between original model and its copy:', dev00)
    #outlier_learn_set=samples_unlearn_outliers(zXtrain,glvq, n,0)
    outlier_learn_set={'unlearn_indices':unlearn_indices, 'unlearn_samples':Xtrain.iloc[unlearn_indices],
                 'relearn_indices':relearn_indices, 'relearn_samples':relearn_samples}
   # outlier_learn_set['relearn_labs']=Ytrain[outlier_learn_set['relearn_indices']]
   # outlier_learn_set['unlearn_labs']=Ytrain[outlier_learn_set['unlearn_indices']]
    #outlier_learn_set=samples_unlearn_outliers(Xtrain,Ytrain, n,0)
    unlearn_indices,relearn_indices=outlier_learn_set['unlearn_indices'], outlier_learn_set['relearn_indices']
    unlearn_samples,relearn_samples=outlier_learn_set['unlearn_samples'], outlier_learn_set['relearn_samples']
    unlearn_labs,relearn_labs=Ytrain[unlearn_indices], Ytrain[relearn_indices]
    zXretrain, zXretest=data_norm_log(Xtrain.iloc[relearn_indices], Xtest)
    #Retraining 
    st=time.time()
    ####################################
    glvq_partial1=GLVQ(distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
        solver_type=solver_type, solver_params=solver_params)
    glvq_partial1.fit(zXretrain, relearn_labs)
    #####################################
    elapsed_retrain=(time.time()-st)/60
    #####################################
    dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_partial1)
    print('After retraining: Deviation between original and retrained models:', dev01)
    ###############################################################################################################
    #Unlearning
    st_un=time.time()
    #########################################
    updated_model=unlearn_sample_effect_glvq(glvq_copy, zXtrain.iloc[unlearn_indices], unlearn_labs, training_info)
    #########################################
    elapsed_untrain=(time.time()-st_un)/60
    print('n=%d, Elapsed time diff=%3f-%3f'%(len(unlearn_indices), elapsed_retrain,elapsed_untrain))
    #########################################
    dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
    print('After unlearning: Deviation between original and unlearned models:', dev02)
    dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1,updated_model)
    print('After unlearning: Deviation between retrained and unlearned models:', dev12)
    ###############################################################################################################
    cratio_uo_ur=dev02/dev12
    print('Dev(prots from original and unlearned models)/Dev(prots from retrained and unlearned models)=%0.03f'%cratio_uo_ur)
    ###############################################################################
    # Compare original (0) vs retrained (1)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf01=compare_perf(glvq, glvq_partial1, data_dict,Ytest)
    # Compare original (0) vs unlearned (2)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf02=compare_perf(glvq, updated_model, data_dict,Ytest)
    # Compare retrained (1) vs vs unlearned (2)
    data_dict={'zX_M1':zXretest,'zX_M2':zXretest}
    perf12=compare_perf(glvq_partial1, updated_model, data_dict,Ytest)
    ##############################################################################
    compare_dict={'num_prot': nprots_per_class, 'n':len(unlearn_indices), 'Mapping':'0:original; 1:retrain; 2:unlearn',
        'et_retrain':elapsed_retrain, 'et_unlearn':elapsed_untrain,
        'prot_dev_01': dev01,'prot_dev_02': dev02,'prot_dev_12': dev12, 
        'prot_dev_02by12':cratio_uo_ur, #'prot_dev_02':1-dev02,  'prot_dev_12':1-dev12,
        'dev_nAcc_01': perf01['dev_npreds'],  'dev_nAcc_02': perf02['dev_npreds'],  'dev_nAcc_12': perf12['dev_npreds'],
        'Acc_M0': perf01['M1_acc'], 'Acc_M1': perf12['M1_acc'], 'Acc_M2': perf12['M2_acc'], 
        'AUC_M0': perf01['M1_auc'], 'AUC_M1': perf12['M1_auc'], 'AUC_M2': perf12['M2_auc'], 
       # 'dev_auc_01': perf01['dev_auc'], 'dev_auc_02': perf02['dev_auc'], 'dev_auc_12': perf12['dev_auc']
                 }
    if cint==0: #'dev_auc_M1M2'
        compare_df=pd.DataFrame.from_dict(data=compare_dict, orient='index').T
        cint+=1
    else:
        temp= pd.DataFrame.from_dict(data=compare_dict, orient='index').T
        compare_df=pd.concat([compare_df, temp])
        print(temp)
        cint+=1

0.0001 0
0.001 0
0.01 11463263
0.1 11463263


In [45]:
nopts=[0.001, 0.0002, 0.003, 0.005]
print(dname)
print(np.min(closest_dists), np.median(closest_dists),np.max(closest_dists))
for n in nopts:
    unlearn_indices=np.where(closest_dists<=n)[0]
    relearn_indices, relearn_samples=relearn_unlearn_samples(Xtrain, unlearn_indices,0)
    if (len(unlearn_indices)==0)|(len(relearn_indices)==0):
        print(n, len(unlearn_indices), len(relearn_indices))

criteo
0.002100861989661666 0.0024557131718477396 0.002909624378193771
0.001 0 11463263
0.0002 0 11463263
0.003 11463263 0
0.005 11463263 0


In [19]:
print(compare_df)
resultspath='/'.join(parts)+'/results/'
compare_df.to_csv(resultspath+'%s_outlier_unlearn_retrain_compare_nprot%d%s.csv'%(dname,nprots_per_class, solver_type), index=False, sep='\t')

  num_prot      n                           Mapping et_retrain et_unlearn  \
0        2     10  0:original; 1:retrain; 2:unlearn   0.222089   0.000226   
0        2    127  0:original; 1:retrain; 2:unlearn   0.314243   0.000381   
0        2  15166  0:original; 1:retrain; 2:unlearn     0.1938   0.000593   

  prot_dev_01 prot_dev_02 prot_dev_12 prot_dev_02by12 dev_nAcc_01 dev_nAcc_02  \
0     0.70336     5.75302     5.77789        0.995696        10.0       356.0   
0     0.54938        5.75     5.76861        0.996774         2.0        49.0   
0     1.72667     5.73147     5.49481         1.04307       113.0        33.0   

  dev_nAcc_12  Acc_M0  Acc_M1  Acc_M2  AUC_M0  AUC_M1  AUC_M2  
0       346.0  0.5855  0.5838  0.5269  0.5931  0.5935  0.6905  
0        47.0  0.5855  0.5849  0.5811  0.5931  0.5934  0.6977  
0       -80.0  0.5855  0.5854  0.5874  0.5931  0.6421  0.7233  


In [13]:
from unlearning.unlearn_eval import *
from unlearning.unlearn_lvq import unlearn_relearn_sample_glvq
from utils import samples_unlearn_random, samples_enforce_random
# || Dataset name: Breast cancer data ||
from experiment_utils import dataset_health
from collections import Counter
#dname='breastcancer'
#'adult' #'surgical' # 'diabetes'
dname_all=['diabetes', 'surgical', 'banking', 'adult', 'criteo']
dname=dname_all[4]
Xtrain, Ytrain, Xtest, Ytest, features=dataset_health(dname)
#zXtrain, zXtest=data_normalization(Xtrain, Xtest)
zXtrain, zXtest=data_norm_log(Xtrain, Xtest)
########################################################################################
if (dname=='breastcancer'):
    nopts=[1,5,20,30,50,100]
#nopts=[0.01,0.05,0.1,1,2]
elif (dname=='criteo'):
    nopts=np.ceil(np.array([0.0001, 0.01, 0.1, 0.2])*len(Ytrain)).astype(np.int32)
else:
    nopts=np.ceil(np.array([0.0001, 0.001, 0.01, 0.05, 0.1, 0.2])*len(Ytrain)).astype(np.int32)
###################################################################################
# Model params to compare; 
# Training original model
dist_name, activation_type="squared-euclidean", "identity"
solver_type, solver_params="lbfgs", {"max_runs": 5, "step_size": np.array([0.05]),  "k": 3,
}
print(dname, ' random ', solver_type)
nprots=1
nprots_per_class=nprots
training_info={'setsize':zXtrain.shape[0], 'class_weight':Counter(Ytrain)}
########################################################################################
# Unlearning parameters to compare
# * number of random samples to unlearn n=[1,5,20,30,50]
cint=0
retrained_n, unlearned_n={},{}
for n in nopts:
    retrained_iter, unlearned_iter={},{}
    for iter in [0,1,2]:
        random_learn_set=samples_unlearn_random(Xtrain, Ytrain, n,0) 
        if iter>0:
            glvq_copy.prototypes_=glvq_copy.secure_prototypes_.copy()
        unlearn_indices,relearn_indices=random_learn_set['unlearn_indices'], random_learn_set['relearn_indices']
        unlearn_samples,relearn_samples=random_learn_set['unlearn_samples'], random_learn_set['relearn_samples']
        unlearn_labs,relearn_labs=random_learn_set['unlearn_labs'], random_learn_set['relearn_labs']
        enforce_indices, enforce_samples=samples_enforce_random(Xtrain, unlearn_indices,Ytrain)
        print(dname, 'iternum-num-', iter,'Amt of overlap between unlearn and enforce indices:', len(np.intersect1d(unlearn_indices, enforce_indices)))


criteo  random  lbfgs
criteo iternum-num- 0 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 1 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 2 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 0 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 1 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 2 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 0 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 1 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 2 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 0 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 1 Amt of overlap between unlearn and enforce indices: 0
criteo iternum-num- 2 Amt of overlap between unlearn and enforce indices: 0


In [24]:
closest_dists

array([0.00570646, 0.02083404, 0.00168147, ..., 0.00038188, 0.00078343,
       0.0046479 ])

In [95]:
rel_distances

array([[  12.59533194, 1593.54611123, 1593.54267533],
       [   8.2208417 , 1614.84381882, 1614.8400971 ],
       [  32.14440009, 1567.15114384, 1567.14938803],
       ...,
       [   6.66853435, 1641.59708184, 1641.59385068],
       [  45.13539977, 1735.94034003, 1735.93769355],
       [  20.77454422, 1426.24194148, 1426.239213  ]])

In [21]:
%autoreload 2
from utils import samples_unlearn_random
from unlearning.unlearn_eval import *
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq

dist_name, activation_type="squared-euclidean", "identity"
solver_type, solver_params="wgd", {"max_runs": 5, "k": 3, "step_size": np.array([0.05])}
nprots_per_class=3
glvq=model = GLVQ(
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)

glvq.fit(zXtrain, Ytrain)

########################################################################################
# Unlearning parameters to compare
# * number of random samples to unlearn n=[1,5,20,30,50]

for n in [1,5,20,30,50,100]:
    glvq_copy=GLVQ(
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy.fit(zXtrain, Ytrain)
    dev00, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
    print('Before unlearning: Fidelity between original model and its copy:', 1-dev00)
    random_learn_set=samples_unlearn_random(Xtrain,Ytrain, n,0)
    unlearn_indices,relearn_indices=random_learn_set['unlearn_indices'], random_learn_set['relearn_indices']
    unlearn_samples,relearn_samples=random_learn_set['unlearn_samples'], random_learn_set['relearn_samples']
    unlearn_labs,relearn_labs=random_learn_set['unlearn_labs'], random_learn_set['relearn_labs']
    zXretrain, zXretest=data_normalization(Xtrain.iloc[relearn_indices], Xtest)
    st=time.time()
    glvq_partial1=GLVQ(distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
        solver_type=solver_type, solver_params=solver_params)
    glvq_partial1.fit(zXretrain, relearn_labs)
    elapsed_retrain=(time.time()-st)/60
    st_un=time.time()
    updated_prots=unlearn_sample_effect_glvq(glvq, zXtrain.iloc[unlearn_indices], unlearn_labs)
    elapsed_untrain=(time.time()-st_un)/60
    glvq_copy.set_prototypes(updated_prots)
    dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_copy)
    dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1,glvq_copy)
    dev02, max_dev_indx02=compare_fidelity_glvq(glvq, glvq_copy)
    print('After unlearning: Fidelity between retrained and unlearned moodels:', 1-dev12)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf01=compare_perf(glvq, glvq_partial1, data_dict,Ytest)
    
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf02=compare_perf(glvq, glvq_copy, data_dict,Ytest)
    
    data_dict={'zX_M1':zXretest,'zX_M2':zXretest}
    perf12=compare_perf(glvq_partial1, glvq_copy, data_dict,Ytest)
    compare_dict={'n':n,'Mapping':'0:original; 1:retrain; 2:unlearn',
        'prot_dev_01':1-dev01, 'prot_dev_02':1-dev02,  'prot_dev_12':1-dev12,
        'dev_npreds_01': perf01['dev_npreds'],  'dev_npreds_02': perf02['dev_npreds'],  'dev_npreds_12': perf12['dev_npreds'],
        'dev_acc_01': perf01['dev_acc'], 'dev_acc_02': perf02['dev_acc'], 'dev_acc_12': perf12['dev_acc'], 
        'dev_auc_01': perf01['dev_auc'], 'dev_auc_02': perf02['dev_auc'], 'dev_auc_12': perf12['dev_auc']}
    print('n=%d, time diff=%3f-%3f'%(n, elapsed_retrain,elapsed_untrain))
    if n==1: #'dev_auc_M1M2'
        compare_df=pd.DataFrame.from_dict(data=compare_dict, orient='index').T
    else:
        compare_df=pd.concat([compare_df, 
                             pd.DataFrame.from_dict(data=compare_dict, orient='index').T])
compare_df.insert(0,'num_prot', nprots_per_class)

compare_df

Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.9570325405058471
n=1, time diff=0.031552-0.000383
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.955020863006206
n=5, time diff=0.030930-0.000469
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.88257557988577
n=20, time diff=0.031545-0.000615
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.8626778575086758
n=30, time diff=0.032176-0.000745
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.8868824043623249
n=50, time diff=0.027314-0.000787
Before unlearning: Fidelity between original model and 

,num_prot,n,Mapping,prot_dev_01,prot_dev_02,prot_dev_12,dev_npreds_01,dev_npreds_02,dev_npreds_12,dev_acc_01,dev_acc_02,dev_acc_12,dev_auc_01,dev_auc_02,dev_auc_12
0,3,1,0:original; 1:retrain; 2:unlearn,0.979945,0.979945,0.957033,4696,4339,6003,0.000579,-0.002905,-0.003484,0.00418,0.007115,0.002935
0,3,5,0:original; 1:retrain; 2:unlearn,0.981692,0.981692,0.955021,4624,3481,6713,0.001488,-0.001968,-0.003456,0.003091,0.002451,-0.00064
0,3,20,0:original; 1:retrain; 2:unlearn,0.899237,0.899237,0.882576,3510,17456,18027,0.000828,0.018603,0.017776,0.001388,0.024831,0.023443
0,3,30,0:original; 1:retrain; 2:unlearn,0.873257,0.873257,0.862678,4569,17315,19048,0.000527,0.010208,0.009681,0.003171,0.018993,0.015822
0,3,50,0:original; 1:retrain; 2:unlearn,0.900429,0.900429,0.886882,4302,13704,14113,0.000757,0.012609,0.011851,0.001273,0.007967,0.006694
0,3,100,0:original; 1:retrain; 2:unlearn,0.935408,0.935408,0.915804,5201,10781,12983,-0.000243,0.007323,0.007566,0.004353,0.001282,-0.003071


In [8]:
a=3.09678504
np.round(a, 3)

3.097

In [9]:
np.round(a, 4)

3.0968